# Krish Naik Agentic AI 3.0 Course Langchain-Runtime

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [34]:
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain.messages import SystemMessage,HumanMessage,ToolMessage
from langchain.agents.middleware import before_model,before_agent,after_model, after_agent,wrap_model_call,wrap_tool_call
from langchain.chat_models import init_chat_model
from langchain.agents.middleware import AgentState

from langgraph.runtime import Runtime
from langchain.tools import ToolRuntime
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from dataclasses import dataclass
from typing import Any,Callable

from langchain.agents.middleware import dynamic_prompt

## Agent State

In [19]:
class MyAgentState(AgentState):
    user_name: str

@tool
def get_weather(location: str) -> str:
    """Get the weather for a location."""
    print(f"[TOOL] get_weather({location})")
    return f"The weather in {location} is sunny."

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    state_schema=MyAgentState,
)
result = agent.invoke({"messages": [{ "role": "user", "content": "What is the weather in Tokyo?"} ],
    "user_name": "John"
})
from rich import print
print(result)

[TOOL] get_weather(Tokyo)

{
    'messages': [
        HumanMessage(
            content='What is the weather in Tokyo?',
            additional_kwargs={},
            response_metadata={},
            id='4cd52b39-d105-48aa-b09e-132a3e1ccf9a'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 23,
                    'prompt_tokens': 132,
                    'total_tokens': 155,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wGAGNKtkphO8FNQBASFUNV1gCC',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-7445-75e1-9d44-82d6ca703401-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_vtrFizK1IJEu75FoylgDng4s',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 132,
                'output_tokens': 23,
                'total_tokens': 155,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='The weather in Tokyo is sunny.',
            name='get_weather',
            id='fd1a53aa-dc96-4937-8f5b-9396444dc34e',
            tool_call_id='call_vtrFizK1IJEu75FoylgDng4s'
        ),
        AIMessage(
            content="It's sunny in Tokyo right now. Would you like more details (temperature, humidity, hourly 
forecast, or a multi-day outlook)?",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 99,
                    'prompt_tokens': 167,
                    'total_tokens': 266,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wIZPmc9UgDaW8InDgBySDpHQ7L',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-7951-7771-bdd8-927f055ab9c7-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 167,
                'output_tokens': 99,
                'total_tokens': 266,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        )
    ],
    'user_name': 'John'
}

## Agent context and Tool runtime

In [20]:
class MyAgentState(AgentState):
    user_name: str


@dataclass
class MyCustomerContext:
    user_id: str

loyalty_store=InMemoryStore()
checkpointer = InMemorySaver()

@tool
def save_customer_preferences(preference: str, runtime: ToolRuntime[MyCustomerContext]) -> str:
    """Save a customer's preference to long-term memory."""
    print(f' save_customer_preferences runtime.state={runtime.state}')
    user_id = runtime.context.user_id
    print(f'save_customer_preferences called for {user_id} with incoming {preference}')
    # Look for existing customer record
    existing = runtime.store.get( ("users"),user_id)

    if existing:
       
        data = existing.value
    else:
        data = {}

    print(f'saving preference {preference} for {user_id}')
    # Add/update preference
    data["preferences"] = preference

    # Save it
    runtime.store.put( ("users"), user_id,data)
    return f"Saved preference: {preference}"

    
@tool
def fetch_customer_preferences(runtime: ToolRuntime[MyCustomerContext])-> str:
  "Fetch the customer's Saved Preferences from a long term Memory"
  print(f' fetch_customer_preferences runtime.state={runtime.state}')
  user_id=runtime.context.user_id
  print(f'fetch_customer_preferences called for {user_id}')
  preferences = "No Preferences"

  if runtime.store:
    if memory := runtime.store.get(('users'),user_id):
      preferences = memory.value['preferences']

  return preferences

pref_agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[save_customer_preferences,fetch_customer_preferences],
    context_schema=MyCustomerContext,
    store=loyalty_store,
    checkpointer = checkpointer,
    state_schema=MyAgentState
)

config = {"configurable": {"thread_id": "pref-agent-demo"}}

user_id = "aj123"
dialogue = [
   "What's my fav movie?",
    "I am AJ, I like movie interstellar",
   "What's my fav movie?",
]
state_value = {'user_name': 'Ajuser'}
for d in dialogue: 
    result = pref_agent.invoke(
        {"messages": [{"role": "user", "content": d}] ,
        "user_name" :  'Ajuser'
        },
        context=MyCustomerContext(user_id=user_id),   # injected at invocation time
        config = config
    )

from rich import print
print(result)

## Another user
user_id = "priya123"
result = pref_agent.invoke(
        {"messages": [{"role": "user", "content": "What's my fav movie?"}],
        "user_name" :  'priyauser'
        },
        context=MyCustomerContext(user_id=user_id),   # injected at invocation time
        config = config,
)
print(result)

fetch_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})], 'user_name': 'Ajuser'}

fetch_customer_preferences called for aj123

save_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content='No Preferences', 
name='fetch_customer_preferences', id='2d983da9-656b-4401-bc56-501668e15bff', 
tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'), AIMessage(content="I don't know yet — I couldn't find a saved 
favorite movie for you. What is your favorite movie? If you tell me, I can remember it for next time. If you're not
sure, tell me a genre or a few movies you like and I can suggest likely favorites.", additional_kwargs={'refusal': 
None}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 181, 'total_tokens': 502, 
'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 
'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 
'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': 
None, 'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh', 'service_tier': 'default', 'finish_reason': 'stop', 
'logprobs': None}, id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0', tool_calls=[], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 181, 'output_tokens': 321, 'total_tokens': 502, 'input_token_details': {'audio': 0,
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}}), HumanMessage(content='I am AJ, I like 
movie interstellar', additional_kwargs={}, response_metadata={}, id='500d1b38-5e4a-44fd-9208-6754a2fb691e'), 
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 
159, 'prompt_tokens': 256, 'total_tokens': 415, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 
'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': 
{'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wsctjdbL30P84pOYZjKiPrR65J', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f97-0820-7e41-af28-5bf906cd6e69-0', tool_calls=[{'name': 'save_customer_preferences', 'args': 
{'preference': 'AJ: favorite movie: Interstellar'}, 'id': 'call_HJEL49lJmg4231OtkMoXLBv7', 'type': 'tool_call'}], 
invalid_tool_calls=[], usage_metadata={'input_tokens': 256, 'output_tokens': 159, 'total_tokens': 415, 
'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}})], 
'user_name': 'Ajuser'}

save_customer_preferences called for aj123 with incoming AJ: favorite movie: Interstellar

saving preference AJ: favorite movie: Interstellar for aj123

fetch_customer_preferences runtime.state={'messages': [HumanMessage(content="What's my fav movie?", 
additional_kwargs={}, response_metadata={}, id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'), AIMessage(content='', 
additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 20, 'prompt_tokens': 
153, 'total_tokens': 173, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 
'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 
'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0', tool_calls=[{'name': 'fetch_customer_preferences', 'args': {},
'id': 'call_klf0D6PklcTsYCRHrT2F3eFX', 'type': 'tool_call'}], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 153, 'output_tokens': 20, 'total_tokens': 173, 'input_token_details': {'audio': 0, 
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}), ToolMessage(content='No Preferences', 
name='fetch_customer_preferences', id='2d983da9-656b-4401-bc56-501668e15bff', 
tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'), AIMessage(content="I don't know yet — I couldn't find a saved 
favorite movie for you. What is your favorite movie? If you tell me, I can remember it for next time. If you're not
sure, tell me a genre or a few movies you like and I can suggest likely favorites.", additional_kwargs={'refusal': 
None}, response_metadata={'token_usage': {'completion_tokens': 321, 'prompt_tokens': 181, 'total_tokens': 502, 
'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 256, 
'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 
'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-mini-2025-08-07', 'system_fingerprint': 
None, 'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh', 'service_tier': 'default', 'finish_reason': 'stop', 
'logprobs': None}, id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0', tool_calls=[], invalid_tool_calls=[], 
usage_metadata={'input_tokens': 181, 'output_tokens': 321, 'total_tokens': 502, 'input_token_details': {'audio': 0,
'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 256}}), HumanMessage(content='I am AJ, I like 
movie interstellar', additional_kwargs={}, response_metadata={}, id='500d1b38-5e4a-44fd-9208-6754a2fb691e'), 
AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 
159, 'prompt_tokens': 256, 'total_tokens': 415, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 
'audio_tokens': 0, 'reasoning_tokens': 128, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': 
{'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 
'gpt-5-mini-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EG5wsctjdbL30P84pOYZjKiPrR65J', 
'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, 
id='lc_run--01a02f97-0820-7e41-af28-5bf906cd6e69-0', tool_calls=[{'name': 'save_customer_preferences', 'args': 
{'preference': 'AJ: favorite movie: Interstellar'}, 'id': 'call_HJEL49lJmg4231OtkMoXLBv7', 'type': 'tool_call'}], 
invalid_tool_calls=[], usage_metadata={'input_tokens': 256, 'output_tokens': 159, 'total_tokens': 415, 
'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 128}}), 
ToolMessage(content='Saved preference: AJ: favorite movie: Interstellar', name='save_customer_preferences', 
id='ae2174da-351b-4e28-b982-59f3bfa52d7c', tool_call_id='call_HJEL49lJmg4231OtkMoXLBv7'), AIMessage(content='Got 
it, AJ — I saved Interstellar as your favorit

fetch_customer_preferences called for aj123

{
    'messages': [
        HumanMessage(
            content="What's my fav movie?",
            additional_kwargs={},
            response_metadata={},
            id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 153,
                    'total_tokens': 173,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0',
            tool_calls=[
                {
                    'name': 'fetch_customer_preferences',
                    'args': {},
                    'id': 'call_klf0D6PklcTsYCRHrT2F3eFX',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 153,
                'output_tokens': 20,
                'total_tokens': 173,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No Preferences',
            name='fetch_customer_preferences',
            id='2d983da9-656b-4401-bc56-501668e15bff',
            tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'
        ),
        AIMessage(
            content="I don't know yet — I couldn't find a saved favorite movie for you. What is your favorite 
movie? If you tell me, I can remember it for next time. If you're not sure, tell me a genre or a few movies you 
like and I can suggest likely favorites.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 321,
                    'prompt_tokens': 181,
                    'total_tokens': 502,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 181,
                'output_tokens': 321,
                'total_tokens': 502,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        HumanMessage(
            content='I am AJ, I like movie interstellar',
            additional_kwargs={},
            respo

{
    'messages': [
        HumanMessage(
            content="What's my fav movie?",
            additional_kwargs={},
            response_metadata={},
            id='539a1e29-9d10-4262-a7cd-d7c58f45b88c'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 20,
                    'prompt_tokens': 153,
                    'total_tokens': 173,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 0,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wl9abQCin3gJJyrQ26LnCIvnvb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02f96-ec95-7aa2-9367-cfba60763cee-0',
            tool_calls=[
                {
                    'name': 'fetch_customer_preferences',
                    'args': {},
                    'id': 'call_klf0D6PklcTsYCRHrT2F3eFX',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 153,
                'output_tokens': 20,
                'total_tokens': 173,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 0}
            }
        ),
        ToolMessage(
            content='No Preferences',
            name='fetch_customer_preferences',
            id='2d983da9-656b-4401-bc56-501668e15bff',
            tool_call_id='call_klf0D6PklcTsYCRHrT2F3eFX'
        ),
        AIMessage(
            content="I don't know yet — I couldn't find a saved favorite movie for you. What is your favorite 
movie? If you tell me, I can remember it for next time. If you're not sure, tell me a genre or a few movies you 
like and I can suggest likely favorites.",
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 321,
                    'prompt_tokens': 181,
                    'total_tokens': 502,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 256,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG5wm69FIxdI0p2PMWDXZAuybaljh',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02f96-f139-7293-8679-8e3e526eb766-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 181,
                'output_tokens': 321,
                'total_tokens': 502,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 256}
            }
        ),
        HumanMessage(
            content='I am AJ, I like movie interstellar',
            additional_kwargs={},
            respo

In [33]:
class MyAgentState(AgentState):
    logged_in_user: str
    
@dataclass
class MyContext:
    user_id: str

@tool
def get_weather(location: str, runtime: ToolRuntime) -> str:
    """get weather"""
    print(f"\nget_weather {location}")
    return f"Weather is sunny 22F for {location}"

@before_agent
def my_before_agent(state: AgentState, runtime: Runtime)-> dict[str, Any] | None:
    print("[ my_before_agent ]",state, runtime)
    return None

@wrap_model_call
def my_wrap_model_call(request, handler):
    print("[ my_wrap_model_call ] ",request.state, request.runtime)
    return handler(request)

agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    middleware = [ my_before_agent, my_wrap_model_call],
    context_schema=MyContext,
    store=InMemoryStore(),
    state_schema=MyAgentState
)

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")], "logged_in_user": "aj123"})
from rich import print
print(result)

[ my_before_agent ]
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-67f0-6f02-8000-3775da90c4b5',
        checkpoint_ns='my_before_agent.before_agent:9b8162a5-67fe-575d-725b-762450485ccc',
        task_id='9b8162a5-67fe-575d-725b-762450485ccc',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507304.1817842
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

[ my_wrap_model_call ] 
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-6810-6780-8001-64acaa6c56b2',
        checkpoint_ns='model:c2c26a83-c52c-e573-49c2-cccac2d2a2f9',
        task_id='c2c26a83-c52c-e573-49c2-cccac2d2a2f9',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507304.194535
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

get_weather Tokyo

[ my_wrap_model_call ] 
{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 130,
                    'total_tokens': 217,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6auohR7XjC1fvARtEZA9527Lpdb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02fbc-e713-7372-863d-3bf1fd0fdc0f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_Q2cGfMiSszz7vqVZGPJkXNHp',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 130,
                'output_tokens': 87,
                'total_tokens': 217,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Weather is sunny 22F for Tokyo',
            name='get_weather',
            id='875cceee-b6cb-48d2-81ef-ecb6cee09609',
            tool_call_id='call_Q2cGfMiSszz7vqVZGPJkXNHp'
        )
    ],
    'logged_in_user': 'aj123'
}
Runtime(
    context=None,
    store=<langgraph.store.memory.InMemoryStore object at 0x1192c9f40>,
    stream_writer=<function Pregel.stream.<locals>.stream_writer at 0x1192a2020>,
    heartbeat=<function _no_op_heartbeat at 0x1166574c0>,
    previous=None,
    execution_info=ExecutionInfo(
        checkpoint_id='1f19f1ad-7bcf-6f28-8003-8f727ca90e31',
        checkpoint_ns='model:a170bfd7-0f5c-bb43-4cd1-c7042d6f5b49',
        task_id='a170bfd7-0f5c-bb43-4cd1-c7042d6f5b49',
        thread_id=None,
        run_id=None,
        node_attempt=1,
        node_first_attempt_time=1787507306.265217
    ),
    server_info=None,
    control=<langgraph.runtime.RunControl object at 0x118c6ef20>
)

{
    'messages': [
        HumanMessage(
            content='what is the weather in tokyo ? ',
            additional_kwargs={},
            response_metadata={},
            id='2270972d-fee4-4462-a271-71d7d11b344e'
        ),
        AIMessage(
            content='',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 87,
                    'prompt_tokens': 130,
                    'total_tokens': 217,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 64,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6auohR7XjC1fvARtEZA9527Lpdb',
                'service_tier': 'default',
                'finish_reason': 'tool_calls',
                'logprobs': None
            },
            id='lc_run--01a02fbc-e713-7372-863d-3bf1fd0fdc0f-0',
            tool_calls=[
                {
                    'name': 'get_weather',
                    'args': {'location': 'Tokyo'},
                    'id': 'call_Q2cGfMiSszz7vqVZGPJkXNHp',
                    'type': 'tool_call'
                }
            ],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 130,
                'output_tokens': 87,
                'total_tokens': 217,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 64}
            }
        ),
        ToolMessage(
            content='Weather is sunny 22F for Tokyo',
            name='get_weather',
            id='875cceee-b6cb-48d2-81ef-ecb6cee09609',
            tool_call_id='call_Q2cGfMiSszz7vqVZGPJkXNHp'
        ),
        AIMessage(
            content='Right now in Tokyo it’s sunny with a temperature of 22°F (about −5.6°C). \n\nIf you want, I 
can:\n- give a multi-day forecast\n- show hourly temperatures or precipitation chances\n- convert to a different 
unit or compare to your local weather\n\nWhich would you like?',
            additional_kwargs={'refusal': None},
            response_metadata={
                'token_usage': {
                    'completion_tokens': 264,
                    'prompt_tokens': 166,
                    'total_tokens': 430,
                    'completion_tokens_details': {
                        'accepted_prediction_tokens': 0,
                        'audio_tokens': 0,
                        'reasoning_tokens': 192,
                        'rejected_prediction_tokens': 0
                    },
                    'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
                },
                'model_provider': 'openai',
                'model_name': 'gpt-5-mini-2025-08-07',
                'system_fingerprint': None,
                'id': 'chatcmpl-EG6awIfat5QGv1bE9CWwGhMgqiBfK',
                'service_tier': 'default',
                'finish_reason': 'stop',
                'logprobs': None
            },
            id='lc_run--01a02fbc-ef2a-7190-bcdc-e2b7792a55b7-0',
            tool_calls=[],
            invalid_tool_calls=[],
            usage_metadata={
                'input_tokens': 166,
                'output_tokens': 264,
                'total_tokens': 430,
                'input_token_details': {'audio': 0, 'cache_read': 0},
                'output_token_details': {'audio': 0, 'reasoning': 192}
            }
        )
    ],
    'logged_in_user': 'aj123'
}

## Dynamic prompting

In [39]:
@dataclass
class WeatherContext:
    country: str


@tool
def get_weather(location: str) -> str:
    """Get the current weather for a location."""
    return f"The weather in {location} is 75 degrees."


@dynamic_prompt
def weather_prompt(request):
    country = request.runtime.context.country

    print(" DYNAMIC PROMPT EXECUTED")
    print("country =", country)

    if country == "US":
        prompt = (
            "You are a weather assistant. "
            "Report temperatures in Fahrenheit."
        )
    else:
        prompt = (
            "You are a weather assistant. "
            "Report temperatures in Celsius."
        )

    print("prompt =", prompt)

    return prompt


agent = create_agent(
    model="openai:gpt-5-mini",
    tools=[get_weather],
    context_schema=WeatherContext,
    middleware=[weather_prompt],
)

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")]},context = WeatherContext(country= "US"))
from rich import print
print(result["messages"][-1])

result = agent.invoke({"messages": [("user", "what is the weather in tokyo ? ")]},context = WeatherContext(country= "IND"))
from rich import print
print(result["messages"][-1])

DYNAMIC PROMPT EXECUTED

country = US

prompt = You are a weather assistant. Report temperatures in Fahrenheit.

DYNAMIC PROMPT EXECUTED

country = US

...

prompt = You are a weather assistant. Report temperatures in Celsius.

DYNAMIC PROMPT EXECUTED

country = IND

prompt = You are a weather assistant. Report temperatures in Celsius.

AIMessage(
    content='The current temperature in Tokyo is about 24°C (the source reported 75°F, which converts to ≈24°C). 
Would you like more details (conditions, humidity, wind, or a forecast)?',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 435,
            'prompt_tokens': 190,
            'total_tokens': 625,
            'completion_tokens_details': {
                'accepted_prediction_tokens': 0,
                'audio_tokens': 0,
                'reasoning_tokens': 384,
                'rejected_prediction_tokens': 0
            },
            'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}
        },
        'model_provider': 'openai',
        'model_name': 'gpt-5-mini-2025-08-07',
        'system_fingerprint': None,
        'id': 'chatcmpl-EG6oWR1LlFg4yRT1gORu1P9kznyBM',
        'service_tier': 'default',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--01a02fc9-c6d7-7871-ad64-a4a395b43b9f-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 190,
        'output_tokens': 435,
        'total_tokens': 625,
        'input_token_details': {'audio': 0, 'cache_read': 0},
        'output_token_details': {'audio': 0, 'reasoning': 384}
    }
)